In [32]:
from collections import defaultdict
from itertools import product
import json
import numpy as np
from sentence_transformers import SentenceTransformer, util
import statistics
import torch

from dataset import  CDCR_Dataset

#Load data
ecb_paths = ['../CDCR/data/ECBplus-prep/output_data/train']
cdcr_train = CDCR_Dataset(name="cdcr_train", data_path=ecb_paths,context_method="hsu")


def remove_special_token(token_list):
    return [token for token in token_list if token not in ('[E]','[/E]')]

# Load SBERT model
model = SentenceTransformer('all-mpnet-base-v2')

# list of sentences by topic

In [154]:
dict_sentences_by_topic = defaultdict(dict)
#Create dictionary of sentences by topic
# To compare sentence similarity by these groups
for doc_key, doc in cdcr_train.sentences.items():
    topic_num = doc_key.split("/")[0]
    if cdcr_train.name=='cdcr_train' and topic_num=='34':
        continue

    for index, sentence in doc.items():
        sentence_str = ' '.join(sentence.values()) 
        sentence_full_key = f"{doc_key}/{index}"
        dict_sentences_by_topic[topic_num].update({sentence_full_key:sentence_str})

In [156]:
#TAKES TOO LONG, LOAD RESULTS INSTEAD

#For each topic
dict_sentences_pair_by_topic = {}
for topic,sentences_dict in dict_sentences_by_topic.items():
    print(f'current topic {topic}')
    sentences = list(sentences_dict.values())
    
    #Compute similarity
    if not sentences:
        continue
    sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
    cosine_scores = util.cos_sim(sentence_embeddings, sentence_embeddings)

    list_of_keys = list(sentences_dict.keys())
    # Get the indices of the lower triangular elements
    #Validated that the result is the combination
    indices = torch.tril(torch.ones_like(cosine_scores), diagonal=-1).nonzero()

    # Create the dictionary using the indices
    # Also 
    lower_triangle_dict = {}

    for i, j in indices:
        key_sentence1 = list_of_keys[i.item()]
        key_sentence2 = list_of_keys[j.item()]
        key_pair = (key_sentence1, key_sentence2)
        #Ignore sentences without mentions
        if not cdcr_train.mentions_by_sentence[key_sentence1] or not cdcr_train.mentions_by_sentence[key_sentence2]:
            continue

        similarity_score = round(cosine_scores[i, j].item(), 3)
        if similarity_score<1: #ignores those sentences with similarity=1. Mostly exact citations
            lower_triangle_dict[key_pair] = similarity_score

    dict_sentences_pair_by_topic[topic] = lower_triangle_dict


current topic 1
current topic 10
current topic 11
current topic 13
current topic 14
current topic 16
current topic 18
current topic 19
current topic 20
current topic 21
current topic 22
current topic 23
current topic 24
current topic 25
current topic 26
current topic 27
current topic 28
current topic 29
current topic 3
current topic 30
current topic 31
current topic 32
current topic 33
current topic 36
current topic 37
current topic 38
current topic 39
current topic 4
current topic 41
current topic 43
current topic 44
current topic 6
current topic 7
current topic 8
current topic 9


In [213]:
# Export results
# Convert tuple keys to strings
def convert_keys_to_strings(d):
    new_dict = {}
    for key, value in d.items():
        if isinstance(key, tuple):
            key = '###'.join(map(str, key))  # Convert tuple to a string
        if isinstance(value, dict):
            value = convert_keys_to_strings(value)  # Recursively convert nested dictionaries
        new_dict[key] = value
    return new_dict

nested_dict_strings = convert_keys_to_strings(dict_sentences_pair_by_topic)


json_file_path = 'sentence_similarities.json'

# Export the nested dictionary to JSON
with open(json_file_path, 'w') as json_file:
    json.dump(nested_dict_strings, json_file, indent=4)

print(f'Nested dictionary has been exported to {json_file_path}.')

Nested dictionary has been exported to sentence_similarities.json.


In [159]:
# json_file_path = 'sentence_similarities.json'
# with open(json_file_path, 'r') as json_file:
#     nested_dict_strings = json.load(json_file)

# dict_sentences_pair_by_topic = {}

# for key, inner_dict in nested_dict_strings.items():
#     new_inner_dict = {tuple(k.split('###')): v for k, v in inner_dict.items()}
#     dict_sentences_pair_by_topic[key] = new_inner_dict

## Extract mentions from sentences

In [168]:
def unnest_dictionary(input_dict,key=''):
  
    unnested_dict = {}
    for key, value in input_dict.items():
        if isinstance(value, dict):
            unnested_dict.update(unnest_dictionary(value, key))
        else:
            unnested_dict[key] = value

    return unnested_dict


def create_pairs(dict_sentences_pairs,cdcr_dataset, mention_type):
    dict_positive_pairs_by_topic = defaultdict(dict)
    dict_negative_pairs_by_topic = defaultdict(dict)

    for topic, sentence_pairs in dict_sentences_pairs.items():

        if not sentence_pairs: 
            continue
        
        topic_sentence_similarity_median = statistics.median(list(sentence_pairs.values()))

        for (sentence_key1, sentence_key2), score in sentence_pairs.items():
            mention_list_sentence1 = cdcr_dataset.mentions_by_sentence[sentence_key1]
            mention_list_sentence2 = cdcr_dataset.mentions_by_sentence[sentence_key2]

            #Check all the combinations of mentions for each pair of sentences
            for mention_key1,mention_key2 in product(mention_list_sentence1, mention_list_sentence2):

                mention1 = cdcr_dataset.mentions[mention_key1]
                mention2 = cdcr_dataset.mentions[mention_key2]

                if mention1.type!=mention_type or mention2.type!=mention_type: 
                    continue

                same_coref_chain = mention1.coref_chain == mention2.coref_chain
                new_dict_entry = {(mention_key1,mention_key2):score}
                if same_coref_chain:
                    dict_positive_pairs_by_topic[topic].update(new_dict_entry)
                else:
                    dict_negative_pairs_by_topic[topic].update(new_dict_entry)

            
        #Subsampling
        positive_pairs_scores= list(dict_positive_pairs_by_topic[topic].values())

        #Downsample negative pairs based on positive pairs score's Q3
        third_quartile = np.percentile(positive_pairs_scores, 75)
        dict_negative_pairs_by_topic[topic] = {key: value for key, value in dict_negative_pairs_by_topic[topic].items() if value > third_quartile}
        #Downsample positive pairs based on topic's median
        dict_positive_pairs_by_topic[topic] = {key: value for key, value in dict_positive_pairs_by_topic[topic].items() if value < topic_sentence_similarity_median}
    
    return unnest_dictionary(dict_positive_pairs_by_topic), unnest_dictionary(dict_negative_pairs_by_topic)



In [170]:
len(negative_pairs['event'])

17567

In [214]:
positive_pairs_entities,negative_pairs_entities = create_pairs(dict_sentences_pairs=dict_sentences_pair_by_topic,
                                 cdcr_dataset=cdcr_train, mention_type='entity')

positive_pairs_events,negative_pairs_events = create_pairs(dict_sentences_pairs=dict_sentences_pair_by_topic,
                                 cdcr_dataset=cdcr_train, mention_type='event')

positive_pairs = {'entity':positive_pairs_entities, 'event':positive_pairs_events}
negative_pairs = {'entity':negative_pairs_entities,'event':negative_pairs_events}

positive_pairs_export = convert_keys_to_strings(positive_pairs)
negative_pairs_export = convert_keys_to_strings(negative_pairs)

In [215]:
#Export definite pairs
json_file_path = "positive_pairs.json"

# Export dictionary to JSON file
with open(json_file_path, 'w') as json_file:
    json.dump(positive_pairs_export, json_file, indent=4)

json_file_path = "negative_pairs.json"

# Export dictionary to JSON file
with open(json_file_path, 'w') as json_file:
    json.dump(negative_pairs_export, json_file, indent=4)

In [182]:
## Statistics from pairs

cdcr_train.mentions['9_9ecbplus_NON17210019876807755_1_9_YFNM'].subtopic_id

'9ecbplus'

In [196]:
## Statistics from pairs
count_same_topic = 0
count_different_topic = 0
entity_type = 'event'
for (mention1_key, mention2_key), value in negative_pairs[entity_type].items() | positive_pairs[entity_type].items():
    subtopic1 = cdcr_train.mentions[mention1_key].subtopic_id
    subtopic2 = cdcr_train.mentions[mention2_key].subtopic_id
    if subtopic1 ==subtopic2:
        count_same_topic+=1
    else:
        count_different_topic+=1
    
print(count_same_topic/(count_same_topic+count_different_topic))

0.9465135278004709


In [202]:
print('positive similarity score entities', np.median(list(positive_pairs['entity'].values())))
print('positive similarity score events', np.median(list(positive_pairs['event'].values())))
print('negative similarity score entities', np.median(list(negative_pairs['entity'].values())))
print('negative similarity score events', np.median(list(negative_pairs['event'].values())))

positive similarity score entities 0.379
positive similarity score events 0.366
negative similarity score entities 0.803
negative similarity score events 0.812
